In [1]:
using LowLevelFEM

In [2]:
structured_rect_mesh(lx=2, order=2)

In [3]:
material = Material("surface")

Pp = Problem([material], type=:ScalarField, field=:p, rhs_field=:fp, dim=2)
Pv = Problem([material], type=:VectorField, field=:v, rhs_field=:fv, dim=2)

Problem("structured_rect", :VectorField, 2, 2, Material[Material("surface", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 861, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :v, :fv)

In [4]:
pres0 = BoundaryCondition("rightbottom", problem=Pp, p=0)

BoundaryCondition("rightbottom", Problem("structured_rect", :ScalarField, 2, 1, Material[Material("surface", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 861, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :p, :fp), Dict{Symbol, Union{Function, Number, ScalarField}}(:p => 0))

In [5]:
suppT = BoundaryCondition("top", problem=Pv, vx=0, vy=0)
suppB = BoundaryCondition("bottom", problem=Pv, vx=0, vy=0)

BoundaryCondition("bottom", Problem("structured_rect", :VectorField, 2, 2, Material[Material("surface", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 861, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :v, :fv), Dict{Symbol, Union{Function, Number, ScalarField}}(:vy => 0, :vx => 0))

In [6]:
load_v = LoadCondition("surface", fvx=1.0, fvy=0.0)
fv = loadVector(Pv, [load_v])

gp = loadVector(Pp, [])

F = SystemVector([fv, gp])

SystemVector([0.0002777777777777773, 0.0, 0.0002777777777777746, 0.0, 0.0002777777777777775, 0.0, 0.00027777777777777767, 0.0, 0.0005555555555555548, 0.0  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], nothing, Problem[Problem("structured_rect", :VectorField, 2, 2, Material[Material("surface", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 861, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :v, :fv), Problem("structured_rect", :ScalarField, 2, 1, Material[Material("surface", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 861, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :p, :fp)], [0, 1722])

In [7]:
μ = 1.0

# stabilization
γ = 1e-1          # grad-div ( 1e-2...1e0)
δ = 1e-6          # pressure Laplacian (mesh dependent)

A = ∫((SymGrad(Pv) ⋅ SymGrad(Pv)) * 2μ)
D = ∫(Div(Pv) ⋅ Div(Pv) * γ)

# alternative way
AD = ∫((SymGrad(Pv) ⋅ SymGrad(Pv)) * 2μ + γ * (Div(Pv) ⋅ Div(Pv)))

B = ∫(Div(Pv) ⋅ Pp)
C = ∫(Grad(Pp) ⋅ Grad(Pp) * δ)

K = SystemMatrix([A+D B';
    B -C])

sparse([1, 2, 9, 10, 47, 48, 219, 220, 239, 240  …  1722, 1725, 1774, 1784, 1785, 1804, 2013, 2563, 2581, 2583], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1  …  2583, 2583, 2583, 2583, 2583, 2583, 2583, 2583, 2583, 2583], [1.275555555555557, 0.5250000000000022, -0.062222222222221873, 0.1583333333333332, -0.43555555555555625, -0.6333333333333337, -0.07444444444444422, -0.1583333333333334, -0.384444444444442, 0.6333333333333329  …  1.214306433183765e-17, 3.555555555555556e-7, 3.5555555555555934e-7, 1.0666666666666608e-6, 3.555555555555529e-7, 1.066666666666661e-6, 3.5555555555555124e-7, 1.0666666666666688e-6, 1.0666666666666724e-6, -5.6888888888888825e-6], 2583, 2583)

In [8]:
K[:, :]

2583×2583 SparseArrays.SparseMatrixCSC{Float64, Int64} with 116839 stored entries:
⎡⣿⢟⣉⠙⠿⠷⠶⠶⢾⡋⠯⠽⠸⠸⠶⠆⠆⠆⠶⠰⠰⠰⠶⠆⡆⣋⣻⣏⠻⠷⢾⡿⠿⠷⠶⠶⠶⠶⢶⣋⎤
⎢⣇⠘⢿⣷⡶⠾⠿⠛⠛⣷⡶⠰⠰⠶⠆⠇⠇⠭⠽⠘⠘⠛⠃⠃⠃⠛⢹⢻⣷⠿⠛⣷⠶⠶⠿⠽⠛⠛⠛⠛⎥
⎢⢿⡇⣸⡏⠻⣦⡀⠀⠀⠉⠛⠻⠶⣶⣤⣄⣀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢸⣾⡿⣆⠀⠙⠷⣦⣄⠀⠀⠀⠀⠀⎥
⎢⢸⡇⣿⠃⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠈⠉⠙⠛⠷⢶⣦⣤⣀⡀⠀⠀⢸⣿⠁⠹⣆⠀⠀⠈⠙⠻⣦⣄⡀⠀⎥
⎢⡾⠳⢿⣤⡄⠀⠀⠈⠻⣦⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠉⠉⠛⠻⠶⢾⢿⣤⠀⠹⣦⠀⠀⠀⠀⠀⠙⠻⠶⎥
⎢⣏⡇⢘⡋⣿⡀⠀⠀⠀⠙⢿⣷⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢸⣘⣻⡀⠀⢿⣧⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣒⡂⢰⡆⢸⣧⠀⠀⠀⠀⠀⠙⢿⣷⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢐⣲⣾⡇⠀⠀⢻⣧⠀⠀⠀⠀⠀⠀⎥
⎢⠸⠇⠬⠅⠀⢿⡆⠀⠀⠀⠀⠀⠀⠙⢿⣷⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠸⠯⠅⣧⠀⠀⠀⢿⣧⠀⠀⠀⠀⠀⎥
⎢⠨⠅⡍⡅⠀⠘⣷⠀⠀⠀⠀⠀⠀⠀⠀⠙⢿⣷⣄⠀⠀⠀⠀⠀⠀⠀⠨⢭⠅⢻⠀⠀⠀⠀⢻⣧⠀⠀⠀⠀⎥
⎢⢘⡃⣓⠃⠀⠀⢹⣇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢿⣷⣄⠀⠀⠀⠀⠀⢘⣛⠀⢸⡇⠀⠀⠀⠀⢻⣧⠀⠀⠀⎥
⎢⢐⡂⣶⠀⠀⠀⠈⣿⡄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢿⣷⣄⠀⠀⠀⢐⣲⠀⠈⣇⠀⠀⠀⠀⠀⢻⣧⠀⠀⎥
⎢⠸⠇⠭⠀⠀⠀⠀⠸⣧⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢿⣷⣄⠀⠸⠯⠀⠀⣿⠀⠀⠀⠀⠀⠈⢻⣧⠀⎥
⎢⡬⢩⣭⠀⠀⠀⠀⠀⢻⡆⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢿⣷⣬⣭⠀⠀⢸⡆⠀⠀⠀⠀⠀⠀⢻⣧⎥
⎢⡿⢾⣷⣒⣲⣶⣶⣶⣾⣗⣒⢲⢰⣰⡶⡆⡆⣆⣶⢰⢰⣰⡶⡆⡆⣿⣿⣿⣲⣶⣾⣗⣶⣶⣶⣶⣶⣶⣶⣿⎥
⎢⢿⡆⣽⡟⠻⢯⣅⡀⠀⠛⠛⠺⠾⠿⠥⣥⣥⣁⣀⣀⡀⠀⠀⠀⠀⠀⢸⣾⡿⣯⡀⠛⠿⠯⣭⣁⣀⠀⠀⠀⎥
⎢⣾⡷⢿⣤⣄⠀⠈⠙⠳⣦⣤⣄⠀⠀⠀⠀⠀⠀⠉⠉⠉⠙⠛⠛⠲⠶⢾⢿⣤⠈⠻⣦⡀⠀⠀⠈⠉⠛⠳⠶⎥
⎢⢿⡇⢸⡇⠹⣧⡀⠀⠀⠀⠉⠛⠿⣶⣤⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢸⣿⡿⡇⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⎥
⎢⢸⡇⣟⡇⠀⠙⣷⡀⠀⠀⠀⠀⠀⠀⠉⠛⠿⣶⣤⣀⠀⠀⠀⠀⠀⠀⢸⣿⠇⢻⡀⠀⠀⠈⠻⣦⡀⠀⠀⠀⎥
⎢⢸⡇⣿⠀⠀⠀⠈⢿⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠉⠛⠿⣶⣦⣀⠀⠀⢸⣿⠀⠘⣧⠀⠀⠀⠀⠈⠻⣦⡀⠀⎥
⎣⡼⢳⣿⠀⠀⠀⠀⠈⢻⡆⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠉⠛⠿⣶⣼⣿⠀⠀⢹⡆⠀⠀⠀⠀⠀⠈⠻⣦⎦

In [9]:
v, p = solveField(K, F, support=[pres0, suppT, suppB])

(VectorField(Matrix{Float64}[], [0.0; 0.0; … ; 0.01133977847585946; 0.0015401016781899453;;], [0.0], Int64[], 1, :v2D, Problem("structured_rect", :VectorField, 2, 2, Material[Material("surface", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 861, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :v, :fv)), ScalarField(Matrix{Float64}[], [4.063906335611318; 0.0; … ; 0.0012118016344003335; -0.1379758033976745;;], [0.0], Int64[], 1, :scalar, Problem("structured_rect", :ScalarField, 2, 1, Material[Material("surface", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 861, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :p, :fp)))

In [10]:
showDoFResults(v, name="v", visible=true)
showDoFResults(p, name="p")

openPostProcessor()

XOpenIM() failed
Fontconfig warning: using without calling FcInit()
